In [11]:
# Import required libraries and dependencies
import pandas as pd
import hvplot.pandas
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler 
from pathlib import Path
import numpy as np
import hvplot.pandas  
import holoviews as hv
hv.extension('bokeh')  

In [12]:
# Load the data 
df_market_data = pd.read_csv(
    "Resources/crypto_market_data.csv",
    index_col="coin_id")

# Display 
df_market_data.head(10)

,price_change_percentage_24h,price_change_percentage_7d,price_change_percentage_14d,price_change_percentage_30d,price_change_percentage_60d,price_change_percentage_200d,price_change_percentage_1y
coin_id,,,,,,,
bitcoin,1.08388,7.60278,6.57509,7.67258,-3.25185,83.51840,37.51761
ethereum,0.22392,10.38134,4.80849,0.13169,-12.88890,186.77418,101.96023
tether,-0.21173,0.04935,0.00640,-0.04237,0.28037,-0.00542,0.01954
ripple,-0.37819,-0.60926,2.24984,0.23455,-17.55245,39.53888,-16.60193
bitcoin-cash,2.90585,17.09717,14.75334,15.74903,-13.71793,21.66042,14.49384
binancecoin,2.10423,12.85511,6.80688,0.05865,36.33486,155.61937,69.69195
chainlink,-0.23935,20.69459,9.30098,-11.21747,-43.69522,403.22917,325.13186
cardano,0.00322,13.99302,5.55476,10.10553,-22.84776,264.51418,156.09756
litecoin,-0.06341,6.60221,7.28931,1.21662,-17.23960,27.49919,-12.66408


In [13]:
# Generate summary statistics
df_market_data.describe()

,price_change_percentage_24h,price_change_percentage_7d,price_change_percentage_14d,price_change_percentage_30d,price_change_percentage_60d,price_change_percentage_200d,price_change_percentage_1y
count,41.000000,41.000000,41.000000,41.000000,41.000000,41.000000,41.000000
mean,-0.269686,4.497147,0.185787,1.545693,-0.094119,236.537432,347.667956
std,2.694793,6.375218,8.376939,26.344218,47.365803,435.225304,1247.842884
min,-13.527860,-6.094560,-18.158900,-34.705480,-44.822480,-0.392100,-17.567530
25%,-0.608970,0.047260,-5.026620,-10.438470,-25.907990,21.660420,0.406170
50%,-0.063410,3.296410,0.109740,-0.042370,-7.544550,83.905200,69.691950
75%,0.612090,7.602780,5.510740,4.578130,0.657260,216.177610,168.372510
max,4.840330,20.694590,24.239190,140.795700,223.064370,2227.927820,7852.089700


In [14]:
# Plot your data to see what's in your DataFrame
df_market_data.hvplot.line(
    width=800,
    height=400,
    rot=90
)

:NdOverlay   [Variable]
   :Curve   [coin_id]   (value)

---

### Prepare the Data

In [16]:
# 1. Grab the feature matrix
X = df_market_data.values

# 2. Scale it
scaler = StandardScaler()
crypto_scaled = scaler.fit_transform(X)

# 3. Re-wrap as a DataFrame
scaled_df = pd.DataFrame(
    crypto_scaled,
    index=df_market_data.index,
    columns=df_market_data.columns
)

scaled_df.head()



,price_change_percentage_24h,price_change_percentage_7d,price_change_percentage_14d,price_change_percentage_30d,price_change_percentage_60d,price_change_percentage_200d,price_change_percentage_1y
coin_id,,,,,,,
bitcoin,0.508529,0.493193,0.772200,0.235460,-0.067495,-0.355953,-0.251637
ethereum,0.185446,0.934445,0.558692,-0.054341,-0.273483,-0.115759,-0.199352
tether,0.021774,-0.706337,-0.021680,-0.061030,0.008005,-0.550247,-0.282061
ripple,-0.040764,-0.810928,0.249458,-0.050388,-0.373164,-0.458259,-0.295546
bitcoin-cash,1.193036,2.000959,1.760610,0.545842,-0.291203,-0.499848,-0.270317


In [17]:
# Create a DataFrame with the scaled data
scaled_df = pd.DataFrame(
    crypto_scaled,
    index=df_market_data.index,
    columns=df_market_data.columns
)


scaled_df.head()



,price_change_percentage_24h,price_change_percentage_7d,price_change_percentage_14d,price_change_percentage_30d,price_change_percentage_60d,price_change_percentage_200d,price_change_percentage_1y
coin_id,,,,,,,
bitcoin,0.508529,0.493193,0.772200,0.235460,-0.067495,-0.355953,-0.251637
ethereum,0.185446,0.934445,0.558692,-0.054341,-0.273483,-0.115759,-0.199352
tether,0.021774,-0.706337,-0.021680,-0.061030,0.008005,-0.550247,-0.282061
ripple,-0.040764,-0.810928,0.249458,-0.050388,-0.373164,-0.458259,-0.295546
bitcoin-cash,1.193036,2.000959,1.760610,0.545842,-0.291203,-0.499848,-0.270317


---

### Find the Best Value for k Using the Original Scaled DataFrame.

In [19]:
from sklearn.cluster import KMeans
import pandas as pd

# 1. Create a list with the number of k-values from 1 to 11
k_values = list(range(1, 12))

# 2. Create an empty list to store the inertia values
inertias = []

# 3. Compute inertia for each k
for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=1)
    kmeans.fit(scaled_df)
    inertias.append(kmeans.inertia_)

# 4. Bundle into a DataFrame for easy plotting
elbow_df = pd.DataFrame({
    "k":        k_values,
    "inertia": inertias
})

# 5a. Simple line plot (no markers)
line = elbow_df.hvplot.line(
    x="k",
    y="inertia",
    title="Elbow Curve – Scaled DataFrame",
    xticks=k_values,
    xlabel="Number of clusters (k)",
    ylabel="Inertia",
    height=400,
    width=600
)

# 5b. Overlay a scatter to get markers
dots = elbow_df.hvplot.scatter(
    x="k",
    y="inertia",
    size=6
)

# 6. Display the combined plot
(line * dots).opts(title="Elbow Curve – Scaled DataFrame")



:Overlay
   .Curve.I   :Curve   [k]   (inertia)
   .Scatter.I :Scatter   [k]   (inertia)

In [21]:


# Create an empty list to store the inertia values
inertias = []

# Create a for loop to compute the inertia with each possible value of k
# Inside the loop:
# 1. Create a KMeans model using the loop counter for the n_clusters
# 2. Fit the model to the data using `scaled_df`
# 3. Append the model.inertia_ to the inertia list
for k in range(1, 12):
    model = KMeans(n_clusters=k, random_state=1)
    model.fit(scaled_df)
    inertias.append(model.inertia_)

# Bundle k and inertia into a DataFrame for plotting
elbow_df = pd.DataFrame({
    "k":        list(range(1, 12)),
    "inertia": inertias
})

# Plot the elbow curve (line + markers)
line = elbow_df.hvplot.line(
    x="k", y="inertia",
    xticks=list(range(1, 12)),
    xlabel="Number of clusters (k)",
    ylabel="Inertia",
    title="Elbow Curve – Scaled DataFrame",
    height=400,
    width=600
)
dots = elbow_df.hvplot.scatter(x="k", y="inertia", size=6)

# Display combined plot
(line * dots).opts(title="Elbow Curve – Scaled DataFrame")



:Overlay
   .Curve.I   :Curve   [k]   (inertia)
   .Scatter.I :Scatter   [k]   (inertia)

In [22]:
# Create a dictionary with the data to plot the Elbow curve
elbow_data = {
    "k": k_values,
    "inertia": inertias
}

# Create a DataFrame with the data to plot the Elbow curve
elbow_df = pd.DataFrame(elbow_data)

# (Optional) Quick check of the new DataFrame
elbow_df.head()



,k,inertia
0,1,287.000000
1,2,212.123342
2,3,165.136752
3,4,79.022435
4,5,66.413051


In [23]:
# Plot a line chart with all the inertia values computed with
# the different values of k to visually identify the optimal value for k.
elbow_df.hvplot.line(
    x="k",
    y="inertia",
    xticks=k_values,
    xlabel="Number of clusters (k)",
    ylabel="Inertia",
    title="Elbow Curve – Scaled DataFrame",
    height=400,
    width=600
)


:Curve   [k]   (inertia)

#### Answer the following question: 

**Question:** What is the best value for `k`?

**Answer:** 

---

### Cluster Cryptocurrencies with K-means Using the Original Scaled DataFrame

In [24]:
# Initialize the K-Means model using the best value for k

from sklearn.cluster import KMeans

best_k = 4   # ← replace 4 with your chosen k
kmeans = KMeans(n_clusters=best_k, random_state=1)


In [25]:
# Fit the K-Means model using the scaled DataFrame

kmeans.fit(scaled_df)


KMeans(n_clusters=4, random_state=1)

In [26]:
# Predict the clusters to group the cryptocurrencies using the scaled DataFrame
crypto_clusters = kmeans.predict(scaled_df)


# Print the resulting array of cluster values.
print(crypto_clusters)

[2 2 0 0 2 2 2 2 2 0 0 0 0 2 0 2 0 0 2 0 0 2 0 0 0 0 0 0 2 0 0 0 3 2 0 0 1
 0 0 0 0]


In [27]:
# Create a copy of the scaled DataFrame
clustered_scaled_df = scaled_df.copy()

In [28]:
# Add a new column to the copy of the scaled DataFrame with the predicted clusters
clustered_scaled_df["cluster"] = crypto_clusters


# Display the copy of the scaled DataFrame
clustered_scaled_df.head()

,price_change_percentage_24h,price_change_percentage_7d,price_change_percentage_14d,price_change_percentage_30d,price_change_percentage_60d,price_change_percentage_200d,price_change_percentage_1y,cluster
coin_id,,,,,,,,
bitcoin,0.508529,0.493193,0.772200,0.235460,-0.067495,-0.355953,-0.251637,2
ethereum,0.185446,0.934445,0.558692,-0.054341,-0.273483,-0.115759,-0.199352,2
tether,0.021774,-0.706337,-0.021680,-0.061030,0.008005,-0.550247,-0.282061,0
ripple,-0.040764,-0.810928,0.249458,-0.050388,-0.373164,-0.458259,-0.295546,0
bitcoin-cash,1.193036,2.000959,1.760610,0.545842,-0.291203,-0.499848,-0.270317,2


In [29]:
# Create a scatter plot using hvPlot by setting
# `x="price_change_percentage_24h"` and `y="price_change_percentage_7d"`.
# Color the graph points with the labels found using K-Means and
# add the crypto name in the `hover_cols` parameter to identify
# the cryptocurrency represented by each data point.
clustered_scaled_df.reset_index().hvplot.scatter(
    x="price_change_percentage_24h",
    y="price_change_percentage_7d",
    by="cluster",
    hover_cols=["coin_id"],
    title=f"Crypto Clusters (k={best_k}) – Scaled DataFrame",
    height=450,
    width=700
)

:NdOverlay   [cluster]
   :Scatter   [price_change_percentage_24h]   (price_change_percentage_7d,coin_id)

---

### Optimize Clusters with Principal Component Analysis.

In [30]:
# Create a PCA model instance and set `n_components=3`.
pca = PCA(n_components=3, random_state=1)

In [31]:
# Use the PCA model with `fit_transform` to reduce the original scaled DataFrame
# down to three principal components.


# View the scaled PCA data
pca_data = pca.fit_transform(scaled_df)
pca_columns = ["PC1", "PC2", "PC3"]
pca_df = pd.DataFrame(pca_data, index=scaled_df.index, columns=pca_columns)
pca_df.head()

,PC1,PC2,PC3
coin_id,,,
bitcoin,-0.600667,0.842760,0.461595
ethereum,-0.458261,0.458466,0.952877
tether,-0.433070,-0.168126,-0.641752
ripple,-0.471835,-0.222660,-0.479053
bitcoin-cash,-1.157800,2.041209,1.859715


In [32]:
# Retrieve the explained variance to determine how much information
# can be attributed to each principal component.
explained_variance = pca.explained_variance_ratio_
print("Explained variance by component:", explained_variance)
print("Total explained variance (3 PCs):", explained_variance.sum())

Explained variance by component: [0.3719856  0.34700813 0.17603793]
Total explained variance (3 PCs): 0.8950316570309842


#### Answer the following question: 

**Question:** What is the total explained variance of the three principal components?

**Answer:** 

In [34]:
# Create a new DataFrame with the PCA data.
pca_df = pd.DataFrame(
    pca_data,                     # NumPy 
    index=scaled_df.index,        # Copy 
    columns=["PC1", "PC2", "PC3"] # Name 
)

# Copy the crypto names from the original scaled DataFrame


# Set the coin_id column as index


# Display the scaled PCA DataFrame
pca_df.head()



,PC1,PC2,PC3
coin_id,,,
bitcoin,-0.600667,0.842760,0.461595
ethereum,-0.458261,0.458466,0.952877
tether,-0.433070,-0.168126,-0.641752
ripple,-0.471835,-0.222660,-0.479053
bitcoin-cash,-1.157800,2.041209,1.859715


---

### Find the Best Value for k Using the Scaled PCA DataFrame

In [35]:
# Create a list with the number of k-values from 1 to 11
k_values = list(range(1, 12))

In [ ]:
# Create an empty list to store the inertia values
pca_inertias = []

# Create a for loop to compute the inertia with each possible value of k
# Inside the loop:
# 1. Create a KMeans model using the loop counter for the n_clusters
# 2. Fit the model to the data using `pca_df`
# 3. Append the model.inertia_ to the inertia list
from sklearn.cluster import KMeans

for k in k_values:
    model = KMeans(n_clusters=k, random_state=1)
    model.fit(pca_df)
    pca_inertias.append(model.inertia_)



In [ ]:
# Create a dictionary with the data to plot the Elbow curve
elbow_pca_data = {
    "k":       k_values,
    "inertia": pca_inertias
}

# Create a DataFrame with the data to plot the Elbow curve
import pandas as pd
elbow_pca_df = pd.DataFrame(elbow_pca_data)
elbow_pca_df.head()



,k,inertia
0,1,256.874086
1,2,182.339530
2,3,135.442408
3,4,49.665497
4,5,38.672582


In [38]:
#  Plot a line chart with all the inertia values computed with
#the different values of k to visually identify the optimal value for k.
elbow_pca_df.hvplot.line(
    x="k",
    y="inertia",
    xticks=k_values,
    xlabel="Number of clusters (k)",
    ylabel="Inertia",
    title="Elbow Curve – PCA DataFrame",
    height=400,
    width=600
)


:Curve   [k]   (inertia)

#### Answer the following questions: 

* **Question:** What is the best value for `k` when using the PCA data?

  * **Answer:**
k = 4

* **Question:** Does it differ from the best k value found using the original data?
no, its the same
  * **Answer:** 

### Cluster Cryptocurrencies with K-means Using the Scaled PCA DataFrame

In [41]:
# Initialize the K-Means model using the best value for k
from sklearn.cluster import KMeans

best_k_pca = 4 
kmeans_pca = KMeans(n_clusters=best_k_pca, random_state=1)

In [42]:
# Fit the K-Means model using the PCA data
kmeans_pca.fit(pca_df)

KMeans(n_clusters=4, random_state=1)

In [43]:
# Predict the clusters to group the cryptocurrencies using the scaled PCA DataFrame
pca_clusters = kmeans_pca.predict(pca_df)


# Print the resulting array of cluster values.
print(pca_clusters)

[2 2 0 0 2 2 2 2 2 0 0 0 0 2 0 2 0 0 2 0 0 2 0 0 0 0 0 0 2 0 0 0 3 2 0 0 1
 0 0 0 0]


In [44]:
# Create a copy of the scaled PCA DataFrame
clustered_pca_df = pca_df.copy()

# Add a new column to the copy of the PCA DataFrame with the predicted clusters
clustered_pca_df["cluster"] = pca_clusters

# Display the copy of the scaled PCA DataFrame
clustered_pca_df.head()



,PC1,PC2,PC3,cluster
coin_id,,,,
bitcoin,-0.600667,0.842760,0.461595,2
ethereum,-0.458261,0.458466,0.952877,2
tether,-0.433070,-0.168126,-0.641752,0
ripple,-0.471835,-0.222660,-0.479053,0
bitcoin-cash,-1.157800,2.041209,1.859715,2


In [45]:
# Create a scatter plot using hvPlot by setting
# `x="PC1"` and `y="PC2"`.
# Color the graph points with the labels found using K-Means and
# add the crypto name in the `hover_cols` parameter to identify
# the cryptocurrency represented by each data point.
clustered_pca_df.reset_index().hvplot.scatter(
    x="PC1",
    y="PC2",
    by="cluster",
    hover_cols=["coin_id"],
    title=f"Crypto Clusters (k={best_k_pca}) – PCA Space",
    height=450,
    width=700
)

:NdOverlay   [cluster]
   :Scatter   [PC1]   (PC2,coin_id)

### Visualize and Compare the Results

In this section, you will visually analyze the cluster analysis results by contrasting the outcome with and without using the optimization techniques.

In [46]:
# Composite plot to contrast the Elbow curves
# YOUR CODE HERE!
scaled_elbow_plot = elbow_df.hvplot.line(
    x="k", y="inertia",
    xticks=k_values,
    xlabel="k",
    ylabel="Inertia",
    title="Elbow – Scaled DataFrame",
    height=350,
    width=450
)
pca_elbow_plot = elbow_pca_df.hvplot.line(
    x="k", y="inertia",
    xticks=k_values,
    xlabel="k",
    ylabel="Inertia",
    title="Elbow – PCA DataFrame",
    height=350,
    width=450
)
# Display side-by-side
(scaled_elbow_plot + pca_elbow_plot).opts(
    title="Elbow Curve Comparison",
    shared_axes=False
)

:Layout
   .Curve.I  :Curve   [k]   (inertia)
   .Curve.II :Curve   [k]   (inertia)

In [47]:
# Composite plot to contrast the clusters
scaled_cluster_plot = clustered_scaled_df.reset_index().hvplot.scatter(
    x="price_change_percentage_24h",
    y="price_change_percentage_7d",
    by="cluster",
    hover_cols=["coin_id"],
    legend=False,
    title="Clusters – Raw Feature Space",
    height=350,
    width=450
)
pca_cluster_plot = clustered_pca_df.reset_index().hvplot.scatter(
    x="PC1",
    y="PC2",
    by="cluster",
    hover_cols=["coin_id"],
    legend=False,
    title="Clusters – PCA Space",
    height=350,
    width=450
)
# Display side-by-side
(scaled_cluster_plot + pca_cluster_plot).opts(
    title="Cluster Comparison: Raw vs PCA",
    shared_axes=False
)


:Layout
   .NdOverlay.I  :NdOverlay   [cluster]
      :Scatter   [price_change_percentage_24h]   (price_change_percentage_7d,coin_id)
   .NdOverlay.II :NdOverlay   [cluster]
      :Scatter   [PC1]   (PC2,coin_id)

#### Answer the following question: 

  * **Question:** After visually analyzing the cluster analysis results, what is the impact of using fewer features to cluster the data using K-Means?

  * **Answer:** PCA lets K-means focus on the signal rather than the noise, producing clusters that are equally valid but more distinct and interpretable